# ToxicSpans / LegalQAEval pilot analysis

Two questions, from the pilot runs currently in `Experiment_results_publication/Archived_results/`:

1. **Runtime** per model x reasoning arm x effort.
2. **Wrong-text audit** — is `wrong_text_rate` 0 under constrained decoding, and is
   any non-zero value explained by the model hitting `max_new_tokens`?

Symmetric to `ner_analysis.ipynb`, minus the per-subset section: neither dataset has
UNER's treebank split, batch size is fixed at 1 for both (one post / one passage per
prompt, see the eval scripts' module docstrings), so there is no `bs` axis to pivot on.


In [1]:

import sys, json, glob, re
from pathlib import Path
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
sys.path.insert(0, str(ROOT / "src"))
RES = ROOT / "Experiment_results_publication" / "Archived_results"
assert RES.is_dir(), RES

pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

from utils.span_datasets import load_toxic_spans, load_legalqa

# Full corpus sizes, for the extrapolation -- loaded fresh rather than hardcoded,
# so this stays correct if either dataset's HF snapshot ever changes.
FULL_SIZE = {"toxic_spans": len(load_toxic_spans()), "legalqa": len(load_legalqa())}
print("full corpus sizes:", FULL_SIZE)

csvs = sorted(glob.glob(str(RES / "ToxicSpans/Csv/*.csv"))) + \
       sorted(glob.glob(str(RES / "LegalQAEval/Csv/*.csv")))
frames = []
for f in csvs:
    df = pd.read_csv(f)
    if df.empty:
        continue
    df["task"] = "ToxicSpans" if "/ToxicSpans/" in f else "LegalQAEval"
    df["file"] = Path(f).name
    frames.append(df)

runs = pd.concat(frames, ignore_index=True)
runs["model_short"] = runs["model"].str.split("/").str[-1]
runs["effort"] = runs["reasoning_effort"].fillna("n|a")
runs["arm"] = runs.apply(
    lambda r: f"{'ON' if r['reasoning_enabled'] else 'OFF'}"
              + (f"/{r['effort']}" if r["effort"] != "n|a" else ""), axis=1)

print(f"{len(csvs)} CSV files -> {len(runs)} result rows "
      f"({runs['model_short'].nunique()} models, tasks: {sorted(runs['task'].unique())})")


full corpus sizes: {'toxic_spans': 1000, 'legalqa': 1206}
19 CSV files -> 38 result rows (6 models, tasks: ['LegalQAEval', 'ToxicSpans'])


## 1. Runtime per arm

`elapsed_minute_avg` is the wall time for **one seed**, for **one** `(eval_mode,
processor_class)` combination. A submitted job runs both `unconstrained` and
`constrained`, so the cost of one job is the two rows added together -- that is what
`job_min` below reports. `batch_size` is fixed at 1 for both tasks, kept as a column
only for structural symmetry with the NER notebook.


In [2]:

wide = (runs
    .pivot_table(index=["task", "model_short", "arm", "batch_size", "sampling_strategy",
                        "max_examples", "n_iters"],
                 columns="eval_mode", values="elapsed_minute_avg", aggfunc="mean")
    .reset_index())

for col in ("constrained", "unconstrained"):
    if col not in wide.columns:
        wide[col] = float("nan")

wide["job_min"] = wide["constrained"].fillna(0) + wide["unconstrained"].fillna(0)
wide = wide.rename(columns={"constrained": "cons_min", "unconstrained": "uncons_min",
                            "batch_size": "bs", "sampling_strategy": "sampling",
                            "max_examples": "n_ex", "n_iters": "seeds"})
wide = wide.sort_values(["task", "model_short", "arm"])

print(wide[["task", "model_short", "arm", "bs", "sampling", "n_ex", "seeds",
            "uncons_min", "cons_min", "job_min"]].to_string(index=False))


       task    model_short       arm  bs sampling  n_ex  seeds  uncons_min  cons_min  job_min
LegalQAEval       Qwen3-8B       OFF   1 sampling    25      1       2.081     2.142    4.223
LegalQAEval    Qwen3.8-27B       OFF   1 sampling    25      1       5.402     6.002   11.404
LegalQAEval    Qwen3.8-27B    ON/low   1 sampling    25      1      13.294    12.330   25.624
LegalQAEval gemma-4-31B-it       OFF   1 sampling    25      1       5.719     5.759   11.478
LegalQAEval gemma-4-E2B-it       OFF   1 sampling    25      1       0.506     2.799    3.305
LegalQAEval gemma-4-E2B-it        ON   1 sampling    25      1      11.389    13.220   24.609
LegalQAEval   gpt-oss-120b    ON/low   1 sampling    25      1       3.867     3.582    7.449
LegalQAEval   gpt-oss-120b ON/medium   1 sampling    25      1       7.730     7.797   15.527
LegalQAEval    gpt-oss-20b    ON/low   1 sampling    25      1       2.238     2.473    4.711
LegalQAEval    gpt-oss-20b ON/medium   1 sampling    25     

### Cost per example

Normalising by `n_ex` makes arms with different caps comparable, and is the basis for
the extrapolation. Constrained decoding is the number that matters for the paper.


In [3]:

rate = wide.copy()
rate["sec_per_ex_cons"] = rate["cons_min"] * 60 / rate["n_ex"]
rate["sec_per_ex_job"] = rate["job_min"] * 60 / rate["n_ex"]

print(rate[["task", "model_short", "arm", "n_ex",
            "sec_per_ex_cons", "sec_per_ex_job"]]
      .round(2).to_string(index=False))

print()
print("Constrained sec/example, by model (mean over arms):")
print(rate.pivot_table(index="model_short", columns="task",
                       values="sec_per_ex_cons", aggfunc="mean").round(2).to_string())


       task    model_short       arm  n_ex  sec_per_ex_cons  sec_per_ex_job
LegalQAEval       Qwen3-8B       OFF    25             5.14           10.14
LegalQAEval    Qwen3.8-27B       OFF    25            14.40           27.37
LegalQAEval    Qwen3.8-27B    ON/low    25            29.59           61.50
LegalQAEval gemma-4-31B-it       OFF    25            13.82           27.55
LegalQAEval gemma-4-E2B-it       OFF    25             6.72            7.93
LegalQAEval gemma-4-E2B-it        ON    25            31.73           59.06
LegalQAEval   gpt-oss-120b    ON/low    25             8.60           17.88
LegalQAEval   gpt-oss-120b ON/medium    25            18.71           37.26
LegalQAEval    gpt-oss-20b    ON/low    25             5.94           11.31
LegalQAEval    gpt-oss-20b ON/medium    25            14.31           27.24
 ToxicSpans       Qwen3-8B       OFF    25             2.03            3.93
 ToxicSpans    Qwen3.8-27B    ON/low    25            41.13           83.06
 ToxicSpans 

## 2. Extrapolation to 3 seeds on the full datasets

Same method as the NER notebook, deliberately simple: take each arm's measured
**seconds per example**, multiply by the full corpus size, multiply by 3 seeds.

    hours = sec_per_example x corpus_size x 3 / 3600

**What this assumes, and where it will be wrong.** Cost per example is treated as
constant. It is not: pilot examples are a random draw, and generation time scales with
post/passage length and with how much the model reasons. Treat these as
order-of-magnitude planning numbers, not promises -- and note the estimate covers
**both** eval modes (`unconstrained` + `constrained`), since that is what one job
actually runs.


In [4]:

est = rate.copy()
est["full_n"] = est["task"].map({"ToxicSpans": FULL_SIZE["toxic_spans"], "LegalQAEval": FULL_SIZE["legalqa"]})
est["h_1seed"] = est["sec_per_ex_job"] * est["full_n"] / 3600
est["h_3seed"] = est["h_1seed"] * 3

out = est[["task", "model_short", "arm", "n_ex", "sec_per_ex_job",
           "full_n", "h_1seed", "h_3seed"]].copy()
out = out.sort_values(["task", "h_1seed"], ascending=[True, False])
print("Projected wall-clock hours for ONE job (unconstrained + constrained):")
print(out.round(2).to_string(index=False))

Projected wall-clock hours for ONE job (unconstrained + constrained):
       task    model_short       arm  n_ex  sec_per_ex_job  full_n  h_1seed  h_3seed
LegalQAEval    Qwen3.8-27B    ON/low    25           61.50    1206    20.60    61.81
LegalQAEval gemma-4-E2B-it        ON    25           59.06    1206    19.79    59.36
LegalQAEval   gpt-oss-120b ON/medium    25           37.26    1206    12.48    37.45
LegalQAEval gemma-4-31B-it       OFF    25           27.55    1206     9.23    27.68
LegalQAEval    Qwen3.8-27B       OFF    25           27.37    1206     9.17    27.51
LegalQAEval    gpt-oss-20b ON/medium    25           27.24    1206     9.13    27.38
LegalQAEval   gpt-oss-120b    ON/low    25           17.88    1206     5.99    17.97
LegalQAEval    gpt-oss-20b    ON/low    25           11.31    1206     3.79    11.36
LegalQAEval       Qwen3-8B       OFF    25           10.14    1206     3.40    10.19
LegalQAEval gemma-4-E2B-it       OFF    25            7.93    1206     2.66     

## 3. Wrong-text audit

The claim under test (from `PUBLICATION_PLAN.md`): **under constrained decoding the
wrong-text rate is 0, conditional on reasoning terminating.** Any non-zero value must be
a *truncation* failure -- the model spent its whole `max_new_tokens` budget reasoning and
never emitted an answer -- and never a verbatim-copy violation.

`max_new_tokens` is not stored per JSONL row, so it is taken from the CSVs; it is
constant per model (18,000), which the next cell asserts
rather than assumes.


In [5]:

budget = (runs.groupby("model_short")["max_new_tokens"].agg(["nunique", "max"]))
assert (budget["nunique"] == 1).all(), f"max_new_tokens varies within a model:\n{budget}"
BUDGET = budget["max"].to_dict()
print("token budget per model:", BUDGET)

# No _bsN suffix here (BS is always 1, unlike NER's per-batch-size filenames).
PRED = re.compile(r"^(?P<ds>toxic|legalqa)_(?P<model>[^_]+(?:-[^_]+)*)_think_(?P<think>True|False)"
                  r"_(?P<samp>sampling|greedy)_(?P<mode>constrained|unconstrained)"
                  r"_(?P<cfg>think\d.*?)_(?P<proc>[^_]+(?:\|[^_]+)?)$")

rows = []
for f in sorted(glob.glob(str(RES / "ToxicSpans/Predictions/*.jsonl"))) + \
         sorted(glob.glob(str(RES / "LegalQAEval/Predictions/*.jsonl"))):
    m = PRED.match(Path(f).stem)
    if not m:
        print("UNPARSED filename (skipped):", Path(f).name)
        continue
    g = m.groupdict()
    for line in open(f, encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        rows.append(dict(
            task="ToxicSpans" if "/ToxicSpans/" in f else "LegalQAEval",
            dataset_tag=g["ds"], model=g["model"], mode=g["mode"], cfg=g["cfg"],
            budget=BUDGET.get(g["model"]),
            wrong=r["wrong_text"], ntok=r["num_output_tokens"],
            rtok=r.get("num_reasoning_tokens"), atok=r.get("num_answer_tokens"),
            found_end=r.get("found_reasoning_end"), skipped=r.get("reasoning_skipped"),
            reasoning=r.get("reasoning_enabled"), span_count=r.get("span_count"),
        ))

pred = pd.DataFrame(rows)
pred["hit_cap"] = pred["ntok"] >= pred["budget"]
print(f"\n{len(pred)} prediction rows; budget resolved for {pred['budget'].notna().sum()}")
print(pred.groupby("mode").agg(rows=("wrong", "size"), wrong=("wrong", "sum")).to_string())


token budget per model: {'Qwen3-8B': 18000, 'Qwen3.8-27B': 18000, 'gemma-4-31B-it': 18000, 'gemma-4-E2B-it': 18000, 'gpt-oss-120b': 16000, 'gpt-oss-20b': 16000}



950 prediction rows; budget resolved for 950
               rows  wrong
mode                      
constrained     475      0
unconstrained   475    110


In [6]:

cons = pred[pred["mode"] == "constrained"]
bad = cons[cons["wrong"] == 1]

print(f"CONSTRAINED rows: {len(cons)}   wrong_text: {len(bad)}   "
      f"rate: {100*len(bad)/max(len(cons),1):.2f}%")
print()
if len(bad):
    print("Every constrained wrong_text row, with its explanation:")
    show = bad[["task", "model", "cfg", "ntok", "budget", "hit_cap",
                "rtok", "atok", "found_end", "skipped"]]
    print(show.to_string(index=False))
    print()
    unexplained = bad[~bad["hit_cap"].fillna(False)]
    print(f"  hit the token cap        : {int(bad['hit_cap'].sum())}")
    print(f"  did NOT hit the cap      : {len(unexplained)}   <-- must be 0")
    if len(unexplained):
        print("\n  !! UNEXPLAINED verbatim-copy violations:")
        print(unexplained.to_string(index=False))
else:
    print("No constrained wrong_text rows at all.")


CONSTRAINED rows: 475   wrong_text: 0   rate: 0.00%

No constrained wrong_text rows at all.


### The two failure modes are different things

`unconstrained` wrong-text is **expected** -- it is ordinary paraphrasing, the baseline
failure the paper exists to fix. `constrained` wrong-text should only ever be truncation.
Keeping them in one table would hide exactly the contrast the paper is claiming.


In [7]:

summary = (pred.groupby(["mode", "task"])
           .agg(rows=("wrong", "size"), wrong=("wrong", "sum"),
                hit_cap=("hit_cap", "sum"))
           .assign(wrong_rate_pct=lambda d: (100 * d["wrong"] / d["rows"]).round(2))
           .reset_index())
print(summary.to_string(index=False))

print()
print("Constrained wrong_text broken down by whether reasoning terminated:")
c = pred[pred["mode"] == "constrained"].copy()
c["terminated"] = c["found_end"].fillna(True) | c["skipped"].fillna(False)
print(c.groupby("terminated").agg(rows=("wrong", "size"), wrong=("wrong", "sum")).to_string())
print()
print("^ The paper's claim: wrong_text is 0 wherever reasoning terminated.")


         mode        task  rows  wrong  hit_cap  wrong_rate_pct
  constrained LegalQAEval   250      0        0            0.00
  constrained  ToxicSpans   225      0        0            0.00
unconstrained LegalQAEval   250     62        0           24.80
unconstrained  ToxicSpans   225     48        0           21.33

Constrained wrong_text broken down by whether reasoning terminated:
            rows  wrong
terminated             
False        175      0
True         300      0

^ The paper's claim: wrong_text is 0 wherever reasoning terminated.


## Summary

Fill in after running:

- **Runtime** -- see table 1; `job_min` is the per-job cost (both eval modes, one seed).
- **3-seed projection** -- experiments with these datasets will also run only one seed, the reason is explained in the ner analysis notebook.
- **Wrong text** -- constrained rate, and whether every non-zero case hit `max_new_tokens`.
